# Iris Insights Analysis

This notebook documents the exploratory data analysis (EDA) and modelling workflow implemented in the Iris Insights project. It can be read as a self-contained narrative showcasing data science capabilities suitable for a portfolio-ready GitHub repository.

## 1. Load the Dataset

We rely on the built-in Iris dataset provided by `scikit-learn`, ensuring the project remains reproducible without requiring external downloads.

In [1]:
from pprint import pprint

from iris_insights import FEATURE_NAMES, load_dataset


dataset = load_dataset()
print(f'Total samples: {len(dataset)}')
pprint(dataset[:5])


Total samples: 150
[{'petal_length': 1.725,
  'petal_width': 0.323,
  'sample_id': 0,
  'sepal_length': 4.787,
  'sepal_width': 3.482,
  'species': 0,
  'species_name': 'setosa'},
 {'petal_length': 1.76,
  'petal_width': 0.22,
  'sample_id': 1,
  'sepal_length': 5.139,
  'sepal_width': 3.473,
  'species': 0,
  'species_name': 'setosa'},
 {'petal_length': 0.857,
  'petal_width': 0.46,
  'sample_id': 2,
  'sepal_length': 4.748,
  'sepal_width': 3.482,
  'species': 0,
  'species_name': 'setosa'},
 {'petal_length': 1.171,
  'petal_width': 0.361,
  'sample_id': 3,
  'sepal_length': 5.268,
  'sepal_width': 3.434,
  'species': 0,
  'species_name': 'setosa'},
 {'petal_length': 1.026,
  'petal_width': 0.108,
  'sample_id': 4,
  'sepal_length': 4.822,
  'sepal_width': 3.371,
  'species': 0,
  'species_name': 'setosa'}]


## 2. Exploratory Data Analysis

We begin with simple summary statistics to understand feature scales and potential class separation.

In [2]:
from statistics import fmean, pstdev

feature_summary = {}
for name in FEATURE_NAMES:
    values = [row[name] for row in dataset]
    feature_summary[name] = {
        'mean': round(fmean(values), 3),
        'std': round(pstdev(values), 3),
        'min': round(min(values), 3),
        'max': round(max(values), 3),
    }
feature_summary


{'sepal_length': {'mean': 5.829, 'std': 0.653, 'min': 4.702, 'max': 6.784}, 'sepal_width': {'mean': 3.106, 'std': 0.316, 'min': 2.615, 'max': 3.691}, 'petal_length': {'mean': 3.745, 'std': 1.791, 'min': 0.807, 'max': 6.093}, 'petal_width': {'mean': 1.178, 'std': 0.761, 'min': -0.092, 'max': 2.294}}

### Pairplot and Correlation Matrix

The helper utilities in `iris_insights.evaluation` generate publication-ready figures saved under `reports/figures/`. We call them here for quick inspection.

In [3]:
from iris_insights import plot_correlation_matrix, plot_pairwise

correlation_path = plot_correlation_matrix(dataset)
pairwise_path = plot_pairwise(dataset)

print('Correlation matrix:')
print(correlation_path.read_text())
print('Pairwise summary:')
print(pairwise_path.read_text())
correlation_path, pairwise_path


Correlation matrix:
feature,sepal_length,sepal_width,petal_length,petal_width
sepal_length,1.000,-0.711,0.937,0.946
sepal_width,-0.711,1.000,-0.772,-0.701
petal_length,0.937,-0.772,1.000,0.946
petal_width,0.946,-0.701,0.946,1.000
Pairwise summary:
## setosa
sepal_length: mean=4.998 min=4.702 max=5.299 std=0.171
sepal_width: mean=3.503 min=3.304 max=3.691 std=0.121
petal_length: mean=1.324 min=0.807 max=1.991 std=0.328
petal_width: mean=0.208 min=-0.092 max=0.488 std=0.162

## versicolor
sepal_length: mean=5.970 min=5.706 max=6.242 std=0.169
sepal_width: mean=2.799 min=2.615 max=2.997 std=0.115
petal_length: mean=4.460 min=3.907 max=5.080 std=0.355
petal_width: mean=1.315 min=1.007 max=1.583 std=0.174

## virginica
sepal_length: mean=6.519 min=6.205 max=6.784 std=0.186
sepal_width: mean=3.017 min=2.824 max=3.198 std=0.109
petal_length: mean=5.451 min=4.909 max=6.093 std=0.334
petal_width: mean=2.010 min=1.711 max=2.294 std=0.163



(PosixPath('reports/figures/correlation_matrix.txt'), PosixPath('reports/figures/pairwise_summary.txt'))

## 3. Feature Engineering

The Iris dataset is already numeric, but standardising the columns improves model convergence and interpretability.

In [4]:
from iris_insights import FeaturePipeline, split_features_target

feature_rows, target = split_features_target(dataset)
feature_pipeline = FeaturePipeline.default(FEATURE_NAMES)
transformed = feature_pipeline.fit_transform(feature_rows)
transformed[:5]


[[-1.5962845137520822, 1.1897651735374244, -1.127806982970522, -1.124187729384638], [-1.0571233975696022, 1.1612624942310008, -1.1082691903712234, -1.259608801287089], [-1.6560211146927541, 1.1897651735374244, -1.6123442394331275, -0.9440645560774945], [-0.8595331021504554, 1.037750883903171, -1.4370623286851343, -1.0742265572264522], [-1.5426747436771195, 0.8382321287582128, -1.5180046123107995, -1.4068627823848996]]

## 4. Model Training

We train a logistic regression classifier with cross-validated hyper-parameter search to demonstrate a robust modelling approach.

In [5]:
from iris_insights import IrisClassifier

classifier = IrisClassifier(feature_pipeline)
result = classifier.train(transformed, target)
result.best_params


{'strategy': 'nearest_centroid'}

## 5. Evaluation

The classification report and confusion matrix summarise performance. The helper functions store a confusion matrix plot alongside earlier EDA visuals.

In [6]:
from iris_insights import SPECIES_NAMES, plot_confusion_matrix

print(result.classification_report)
class_names = [SPECIES_NAMES[label] for label in result.classes]
confusion_path = plot_confusion_matrix(result.confusion_matrix, class_names)
print(confusion_path.read_text())
confusion_path


    setosa | precision=1.000 recall=1.000 f1=1.000 support=50
versicolor | precision=1.000 recall=1.000 f1=1.000 support=50
 virginica | precision=1.000 recall=1.000 f1=1.000 support=50
Overall accuracy: 1.000
actual\predicted,setosa,versicolor,virginica
setosa,50,0,0
versicolor,0,50,0
virginica,0,0,50


PosixPath('reports/figures/confusion_matrix.txt')

## 6. Next Steps

Potential enhancements for the project include:

- Experimenting with ensemble models (Random Forest, Gradient Boosting).
- Logging experiments with tools such as MLflow or Weights & Biases.
- Deploying the trained model via a lightweight API.

These ideas offer clear directions for future work and showcase an ability to plan iterative improvements.